In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import json
import os
from datetime import datetime, timedelta

# Open-Meteo
BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

# Create directory for caching API responses
os.makedirs('../data/weather_cache', exist_ok=True)

print("Setup complete.")

Setup complete.


In [3]:
# Load the processed flight data that contains airport information
flight_data = pd.read_csv('../data/flights_processed_v1.csv', parse_dates=['FL_DATE'])

# Check what columns are available
print("Available columns:", flight_data.columns.tolist())

# Extract unique combinations of origin/destination airports and dates
airports = set()
airport_dates = []

# Process origin airports
for _, row in flight_data[['ORIGIN', 'FL_DATE']].drop_duplicates().iterrows():
    airports.add(row['ORIGIN'])
    airport_dates.append({'AIRPORT': row['ORIGIN'], 'FL_DATE': row['FL_DATE']})

# Process destination airports
for _, row in flight_data[['DEST', 'FL_DATE']].drop_duplicates().iterrows():
    airports.add(row['DEST'])
    airport_dates.append({'AIRPORT': row['DEST'], 'FL_DATE': row['FL_DATE']})

# Convert to DataFrame
all_airport_dates = pd.DataFrame(airport_dates).drop_duplicates()

# Print information about the data
print(f"Found {len(all_airport_dates)} unique airport-date combinations")
print(f"Spanning {all_airport_dates['FL_DATE'].min()} to {all_airport_dates['FL_DATE'].max()}")
print(f"Across {len(airports)} unique airports")
# Show sample of the data
display(all_airport_dates.head())

Available columns: ['FL_DATE', 'AIRLINE', 'ORIGIN', 'DEST', 'DISTANCE', 'CRS_DEP_TIME', 'CRS_ARR_TIME', 'CRS_ELAPSED_TIME', 'IS_DELAYED']
Found 408361 unique airport-date combinations
Spanning 2019-01-01 00:00:00 to 2023-08-31 00:00:00
Across 380 unique airports


,AIRPORT,FL_DATE
0,FLL,2019-01-09
1,MSP,2022-11-19
2,DEN,2022-07-22
3,MSP,2023-03-06
4,MCO,2020-02-23


In [4]:
# Functions to fetch and process weather data
import time

# Dictionary of airport coordinates (IATA code -> lat,lon)
airport_coords = {}

# Define coordinates for major airports manually
major_airports = {
    "ATL": (33.6407, -84.4277),  # Atlanta
    "DFW": (32.8998, -97.0403),  # Dallas/Fort Worth
    "DEN": (39.8561, -104.6737), # Denver
    "ORD": (41.9786, -87.9048),  # Chicago O'Hare
    "LAX": (33.9416, -118.4085), # Los Angeles
    "CLT": (35.2144, -80.9473),  # Charlotte
    "LAS": (36.0840, -115.1537), # Las Vegas
    "PHX": (33.4342, -112.0080), # Phoenix
    "MCO": (28.4312, -81.3081),  # Orlando
    "SEA": (47.4502, -122.3088), # Seattle
    "MIA": (25.7932, -80.2906),  # Miami
    "JFK": (40.6413, -73.7781),  # New York JFK
    "EWR": (40.6895, -74.1745),  # Newark
    "SFO": (37.6213, -122.3790), # San Francisco
    "BOS": (42.3656, -71.0096),  # Boston
    "MSP": (44.8848, -93.2223),  # Minneapolis
    "DTW": (42.2162, -83.3554),  # Detroit
    "FLL": (26.0742, -80.1506),  # Fort Lauderdale
    "PHL": (39.8729, -75.2437),  # Philadelphia
    "BWI": (39.1774, -76.6684),  # Baltimore
}
airport_coords.update(major_airports)
print(f"Added coordinates for {len(major_airports)} major airports")

# Function to get cached weather data path
def get_cache_path(airport, date):
    """Generate cache file path for weather data."""
    date_str = date.strftime('%Y-%m-%d') if isinstance(date, pd.Timestamp) else date
    return f"../data/weather_cache/{airport}_{date_str}.json"

# Modify the fetch_weather_data function to use Open-Meteo
def fetch_weather_data(airport, date, force_refresh=False):
    """
    Fetch historical weather data for the given airport and date using Open-Meteo API.
    
    Args:
        airport: IATA airport code
        date: Date for which to fetch weather data
        force_refresh: If True, fetch fresh data even if cache exists
        
    Returns:
        Dictionary with weather data or None if failed
    """
    # Format date if it's a pandas Timestamp
    if isinstance(date, pd.Timestamp):
        date_str = date.strftime('%Y-%m-%d')
    else:
        date_str = date
        date = pd.Timestamp(date)
    
    # Create cache file path
    cache_file = get_cache_path(airport, date)
    
    # Check cache first
    if os.path.exists(cache_file) and not force_refresh:
        try:
            with open(cache_file, 'r') as f:
                data = json.load(f)
            print(f"Loaded cached data for {airport} on {date_str}")
            return data
        except Exception as e:
            print(f"Error loading cache for {airport} on {date_str}: {e}")
    
    # Check if we have coordinates for this airport
    if airport not in airport_coords:
        print(f"No coordinates found for airport {airport}")
        return None
    
    lat, lon = airport_coords[airport]
    
    # Open-Meteo uses a different parameter structure
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': date_str,
        'end_date': date_str,
        'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,snowfall_sum',
        'timezone': 'America/New_York'  # Use appropriate timezone
    }
    
    # Add delay to avoid hitting rate limits
    time.sleep(0.1)
    
    try:
        print(f"Fetching weather data for {airport} on {date_str}")
        response = requests.get(BASE_URL, params=params)
        
        if response.status_code == 200:
            data = response.json()
            
            # Cache the response
            with open(cache_file, 'w') as f:
                json.dump(data, f)
            
            return data
        else:
            print(f"API error for {airport} on {date_str}: {response.status_code} - {response.text}")
            return None
    except Exception as e:
        print(f"Exception fetching data for {airport} on {date_str}: {e}")
        return None

# Test with a known airport instead of a random one
sample_airport = "DEN"  # Use Denver which is in our coordinates list
sample_date = pd.Timestamp('2022-07-22')  # Use a date we know exists
print(f"\nTesting with sample airport {sample_airport} on {sample_date}")

test_result = fetch_weather_data(sample_airport, sample_date, force_refresh=False)
if test_result:
    print("Successfully retrieved weather data!")
    print(f"Data contains keys: {list(test_result.keys())}")
    if 'data' in test_result and len(test_result['data']) > 0:
        print(f"Weather conditions: {test_result['data'][0]['weather'][0]['main']} - {test_result['data'][0]['weather'][0]['description']}")
        print(f"Temperature: {test_result['data'][0]['temp']}°C")
        print(f"Humidity: {test_result['data'][0]['humidity']}%")
else:
    print("Failed to retrieve test weather data. Check API key and connection.")

Added coordinates for 20 major airports

Testing with sample airport DEN on 2022-07-22 00:00:00
Loaded cached data for DEN on 2022-07-22
Successfully retrieved weather data!
Data contains keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily']


In [5]:
# Get all unique airports from both origin and destination
unique_origins = flight_data['ORIGIN'].unique()
unique_dests = flight_data['DEST'].unique() 
unique_airports = np.union1d(unique_origins, unique_dests)

In [6]:
# Optional cell: Geocode missing airport coordinates
# Only run this if you want to fetch coordinates for all airports

import pickle
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# Create coordinates cache file path
coords_cache_file = '../data/airport_coordinates.pkl'

# Load previously cached coordinates if available
if os.path.exists(coords_cache_file):
    try:
        with open(coords_cache_file, 'rb') as f:
            cached_coords = pickle.load(f)
            airport_coords.update(cached_coords)
            print(f"Loaded {len(cached_coords)} airports from coordinates cache")
    except Exception as e:
        print(f"Error loading coordinates cache: {e}")

# Function to lookup airport coordinates using geocoding
def get_airport_coordinates(airport_code, max_retries=3):
    """Look up airport coordinates using geocoding service."""
    if airport_code in airport_coords:
        return airport_coords[airport_code]
    
    geolocator = Nominatim(user_agent="flight_delay_predictor_app")
    
    for attempt in range(max_retries):
        try:
            query = f"{airport_code} airport"
            print(f"Looking up coordinates for {airport_code} (attempt {attempt+1}/{max_retries})")
            
            location = geolocator.geocode(query, timeout=10)
            
            if location:
                print(f"Found coordinates for {airport_code}: {location.latitude}, {location.longitude}")
                return (location.latitude, location.longitude)
            
            if attempt == 0:
                query = f"{airport_code} international airport USA"
                print(f"Trying more specific query: {query}")
                location = geolocator.geocode(query, timeout=10)
                
                if location:
                    print(f"Found coordinates for {airport_code}: {location.latitude}, {location.longitude}")
                    return (location.latitude, location.longitude)
            
            time.sleep(1 + attempt)
            
        except (GeocoderTimedOut, GeocoderServiceError) as e:
            print(f"Geocoding error for {airport_code}: {e}")
            if attempt < max_retries - 1:
                wait_time = 2 + attempt * 2
                print(f"Waiting {wait_time} seconds before retry...")
                time.sleep(wait_time)
            else:
                print(f"Failed to get coordinates for {airport_code} after {max_retries} attempts")
                return None
    
    print(f"Could not find coordinates for {airport_code}")
    return None

# Get all unique airports from the dataset
all_airports = set(unique_airports)
print(f"Found {len(all_airports)} unique airports in the dataset")

# Identify airports missing coordinates
missing_airports = [airport for airport in all_airports if airport not in airport_coords]
print(f"Need to find coordinates for {len(missing_airports)} airports")



Loaded 379 airports from coordinates cache
Found 380 unique airports in the dataset
Need to find coordinates for 1 airports


In [7]:
# WARNING: This cell will take significant time to run
# It geocodes all missing airports but only needs to be run once
# After running, all coordinates will be saved to the cache file

# Loop through missing airports and fetch coordinates
print(f"Starting to fetch coordinates for {len(missing_airports)} airports...")
geocoded_count = 0

for i, airport in enumerate(missing_airports):
    if i % 10 == 0:
        print(f"Processing {i+1}/{len(missing_airports)} missing airports...")
    
    coords = get_airport_coordinates(airport)
    
    if coords:
        airport_coords[airport] = coords
        geocoded_count += 1
        
        # Save progress every 10 airports
        if (i + 1) % 10 == 0 or i == len(missing_airports) - 1:
            print(f"Saving coordinates cache with {len(airport_coords)} airports...")
            with open(coords_cache_file, 'wb') as f:
                pickle.dump(airport_coords, f)
    
    # Respect Nominatim usage policy (max 1 request per second)
    time.sleep(1.5)

print(f"Completed geocoding! Found coordinates for {geocoded_count} out of {len(missing_airports)} missing airports")
print(f"Now have coordinates for {len(airport_coords)} out of {len(all_airports)} total airports")

Starting to fetch coordinates for 1 airports...
Processing 1/1 missing airports...
Looking up coordinates for ATW (attempt 1/3)
Trying more specific query: ATW international airport USA
Looking up coordinates for ATW (attempt 2/3)
Looking up coordinates for ATW (attempt 3/3)
Could not find coordinates for ATW
Completed geocoding! Found coordinates for 0 out of 1 missing airports
Now have coordinates for 379 out of 380 total airports


In [11]:
# Loop through combinations and store results with improved sampling
print("Selecting airports and dates for weather data collection...")

# Filter to airports we have coordinates for
airports_with_coords = [airport for airport in unique_airports if airport in airport_coords]
print(f"Found {len(airports_with_coords)} airports with coordinates out of {len(unique_airports)} total airports")

# Take a sample of dates to avoid excessive API calls
# Sample every 30 days instead of 14 days for fewer API calls
sample_dates = []
date_range = pd.date_range(
    flight_data['FL_DATE'].min(),
    flight_data['FL_DATE'].max(), 
    freq='30D'  # Every 30 days to reduce API calls
)
for date in date_range:
    if date in pd.DatetimeIndex(flight_data['FL_DATE']):
        sample_dates.append(date)

print(f"Selected {len(sample_dates)} sample dates")

# Improved fetch function with exponential backoff
def fetch_with_backoff(airport, date, max_retries=5):
    """Fetch weather data with exponential backoff for rate limits"""
    for attempt in range(max_retries):
        try:
            result = fetch_weather_data(airport, date)
            if result:
                return result
            # If no result but no exception, wait a bit and retry
            wait_time = 2 ** attempt + 1  # 1, 3, 5, 9, 17 seconds
            print(f"No data received. Waiting {wait_time} seconds before retry...")
            time.sleep(wait_time)
        except Exception as e:
            wait_time = 2 ** attempt + 2  # 2, 4, 6, 10, 18 seconds
            print(f"Error: {e}. Waiting {wait_time} seconds before retry...")
            time.sleep(wait_time)
    return None

# Now we can collect weather data for these combinations
weather_data = {}
count = 0
checkpoint_file = '../data/weather_data_checkpoint.json'

# Load checkpoint if exists
if os.path.exists(checkpoint_file):
    try:
        with open(checkpoint_file, 'r') as f:
            checkpoint = json.load(f)
            # Convert string keys back to dates
            weather_data = {k: {pd.Timestamp(dk): dv for dk, dv in v.items()} 
                            for k, v in checkpoint.items()}
        print(f"Loaded checkpoint with data for {len(weather_data)} airports")
    except Exception as e:
        print(f"Error loading checkpoint: {e}")

# Process airports in smaller batches
BATCH_SIZE = 50
all_airports = airports_with_coords.copy()

# Find the last processed airport if resuming
last_processed = None
if weather_data:
    last_processed = list(weather_data.keys())[-1]
    print(f"Resuming after airport: {last_processed}")
    
    # Skip airports we've already processed
    if last_processed in all_airports:
        start_idx = all_airports.index(last_processed) + 1
        all_airports = all_airports[start_idx:]

# Split into batches
airport_batches = [all_airports[i:i+BATCH_SIZE] for i in range(0, len(all_airports), BATCH_SIZE)]
print(f"Processing {len(all_airports)} airports in {len(airport_batches)} batches")

batch_num = 1
print("Collecting weather data...")
try:
    for batch in airport_batches:
        print(f"\nProcessing batch {batch_num}/{len(airport_batches)} with {len(batch)} airports")
        batch_num += 1
        
        for airport in batch:
            print(f"Processing airport: {airport}")
            if airport not in weather_data:
                weather_data[airport] = {}
                
            airport_data = weather_data[airport]
            for date in sample_dates:
                date_str = date.strftime('%Y-%m-%d')
                if date_str in airport_data:
                    print(f"Already have data for {airport} on {date_str}, skipping")
                    continue
                    
                # Use the improved fetch function with backoff
                result = fetch_with_backoff(airport, date)
                if result:
                    airport_data[date] = result
                    count += 1
                else:
                    print(f"Failed to fetch data for {airport} on {date_str} after multiple attempts")
                
                # Always wait at least 1 second between calls
                time.sleep(1)
                
                # Save checkpoint every 20 API calls (more frequent checkpoints)
                if count % 20 == 0:
                    print(f"Saving checkpoint at {count} API calls...")
                    # Convert dates to strings for JSON serialization
                    checkpoint = {k: {str(dk): dv for dk, dv in v.items()} 
                                  for k, v in weather_data.items()}
                    with open(checkpoint_file, 'w') as f:
                        json.dump(checkpoint, f)
                
            weather_data[airport] = airport_data
            
            # Save checkpoint after each airport (to avoid losing work)
            print(f"Saving checkpoint after completing airport {airport}...")
            checkpoint = {k: {str(dk): dv for dk, dv in v.items()} 
                          for k, v in weather_data.items()}
            with open(checkpoint_file, 'w') as f:
                json.dump(checkpoint, f)
            
except KeyboardInterrupt:
    print("Operation interrupted by user. Saving checkpoint...")
    checkpoint = {k: {str(dk): dv for dk, dv in v.items()} 
                  for k, v in weather_data.items()}
    with open(checkpoint_file, 'w') as f:
        json.dump(checkpoint, f)
    print("Checkpoint saved. You can resume later.")
    
print(f"Collected weather data for {len(weather_data)} airports, with {count} total API calls")

# Quick preview of the data
if weather_data:
    print("\nSample of weather data:")
    sample_airport = list(weather_data.keys())[0]
    sample_date = list(weather_data[sample_airport].keys())[0]
    print(f"Weather for {sample_airport} on {sample_date}:")
    print(f"Daily data: {weather_data[sample_airport][sample_date]['daily']}")

Selecting airports and dates for weather data collection...
Found 379 airports with coordinates out of 380 total airports
Selected 57 sample dates
Loaded checkpoint with data for 132 airports
Resuming after airport: SPS
Processing 247 airports in 5 batches

Processing batch 1/5 with 50 airports
Processing airport: ECP
Fetching weather data for ECP on 2019-01-01
API error for ECP on 2019-01-01: 429 - {"reason":"Daily API request limit exceeded. Please try again tomorrow.","error":true}
No data received. Waiting 2 seconds before retry...
Fetching weather data for ECP on 2019-01-01
API error for ECP on 2019-01-01: 429 - {"error":true,"reason":"Daily API request limit exceeded. Please try again tomorrow."}
No data received. Waiting 3 seconds before retry...
Fetching weather data for ECP on 2019-01-01
API error for ECP on 2019-01-01: 429 - {"error":true,"reason":"Daily API request limit exceeded. Please try again tomorrow."}
No data received. Waiting 5 seconds before retry...
Fetching weath

In [24]:
# Convert raw API responses to structured DataFrame
print("Converting weather data to DataFrame format...")

weather_records = []
for airport, dates in weather_data.items():
    for date_str, data in dates.items():
        # Extract the daily weather data
        daily = data['daily']
        
        # Each value is a list with one item (for the single day)
        record = {
            'AIRPORT': airport,
            'FL_DATE': pd.Timestamp(date_str),
            'TEMP_MAX': daily['temperature_2m_max'][0],
            'TEMP_MIN': daily['temperature_2m_min'][0],
            'PRECIPITATION': daily['precipitation_sum'][0],
            'RAIN': daily['rain_sum'][0],
            'SNOW': daily['snowfall_sum'][0]
        }
        weather_records.append(record)

# Create DataFrame from the extracted records
weather_df = pd.DataFrame(weather_records)

# Display a sample of the weather data
print(f"Created weather DataFrame with {len(weather_df)} rows")
display(weather_df.head())

# Check for any missing values
print("\nMissing values in weather data:")
print(weather_df.isnull().sum())

Converting weather data to DataFrame format...
Created weather DataFrame with 8911 rows


,AIRPORT,FL_DATE,TEMP_MAX,TEMP_MIN,PRECIPITATION,RAIN,SNOW
0,FLL,2019-01-01,25.7,22.9,0.4,0.4,0.0
1,FLL,2019-01-15,22.6,13.8,0.0,0.0,0.0
2,FLL,2019-01-29,19.7,9.6,0.0,0.0,0.0
3,FLL,2019-02-12,26.6,22.3,1.2,1.2,0.0
4,FLL,2019-02-26,25.2,21.7,13.2,13.2,0.0



Missing values in weather data:
AIRPORT          0
FL_DATE          0
TEMP_MAX         0
TEMP_MIN         0
PRECIPITATION    0
RAIN             0
SNOW             0
dtype: int64


In [33]:
# Functions to fetch and process weather data
import time

# Dictionary of airport coordinates (IATA code -> lat,lon)
airport_coords = {}

# Define coordinates for major airports manually
major_airports = {
    "ATL": (33.6407, -84.4277),  # Atlanta
    "DFW": (32.8998, -97.0403),  # Dallas/Fort Worth
    "DEN": (39.8561, -104.6737), # Denver
    "ORD": (41.9786, -87.9048),  # Chicago O'Hare
    "LAX": (33.9416, -118.4085), # Los Angeles
    "CLT": (35.2144, -80.9473),  # Charlotte
    "LAS": (36.0840, -115.1537), # Las Vegas
    "PHX": (33.4342, -112.0080), # Phoenix
    "MCO": (28.4312, -81.3081),  # Orlando
    "SEA": (47.4502, -122.3088), # Seattle
    "MIA": (25.7932, -80.2906),  # Miami
    "JFK": (40.6413, -73.7781),  # New York JFK
    "EWR": (40.6895, -74.1745),  # Newark
    "SFO": (37.6213, -122.3790), # San Francisco
    "BOS": (42.3656, -71.0096),  # Boston
    "MSP": (44.8848, -93.2223),  # Minneapolis
    "DTW": (42.2162, -83.3554),  # Detroit
    "FLL": (26.0742, -80.1506),  # Fort Lauderdale
    "PHL": (39.8729, -75.2437),  # Philadelphia
    "BWI": (39.1774, -76.6684),  # Baltimore
}
airport_coords.update(major_airports)
print(f"Added coordinates for {len(major_airports)} major airports")

# Function to get cached weather data path
def get_cache_path(airport, date):
    """Generate cache file path for weather data."""
    date_str = date.strftime('%Y-%m-%d') if isinstance(date, pd.Timestamp) else date
    return f"../data/weather_cache/{airport}_{date_str}.json"

# Modify the fetch_weather_data function to use Open-Meteo
def fetch_weather_data(airport, date, force_refresh=False):
    """
    Fetch historical weather data for the given airport and date using Open-Meteo API.
    
    Args:
        airport: IATA airport code
        date: Date for which to fetch weather data
        force_refresh: If True, fetch fresh data even if cache exists
        
    Returns:
        Dictionary with weather data or None if failed
    """
    # Format date if it's a pandas Timestamp
    if isinstance(date, pd.Timestamp):
        date_str = date.strftime('%Y-%m-%d')
    else:
        date_str = date
        date = pd.Timestamp(date)
    
    # Create cache file path
    cache_file = get_cache_path(airport, date)
    
    # Check cache first
    if os.path.exists(cache_file) and not force_refresh:
        try:
            with open(cache_file, 'r') as f:
                data = json.load(f)
            print(f"Loaded cached data for {airport} on {date_str}")
            return data
        except Exception as e:
            print(f"Error loading cache for {airport} on {date_str}: {e}")
    
    # Check if we have coordinates for this airport
    if airport not in airport_coords:
        print(f"No coordinates found for airport {airport}")
        return None
    
    lat, lon = airport_coords[airport]
    
    # Open-Meteo uses a different parameter structure
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': date_str,
        'end_date': date_str,
        'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,snowfall_sum',
        'timezone': 'America/New_York'  # Use appropriate timezone
    }
    
    # Add delay to avoid hitting rate limits
    time.sleep(0.1)
    
    try:
        print(f"Fetching weather data for {airport} on {date_str}")
        response = requests.get(BASE_URL, params=params)
        
        if response.status_code == 200:
            data = response.json()
            
            # Cache the response
            with open(cache_file, 'w') as f:
                json.dump(data, f)
            
            return data
        else:
            print(f"API error for {airport} on {date_str}: {response.status_code} - {response.text}")
            return None
    except Exception as e:
        print(f"Exception fetching data for {airport} on {date_str}: {e}")
        return None

# Test with a known airport instead of a random one
sample_airport = "DEN"  # Use Denver which is in our coordinates list
sample_date = pd.Timestamp('2022-07-22')  # Use a date we know exists
print(f"\nTesting with sample airport {sample_airport} on {sample_date}")

test_result = fetch_weather_data(sample_airport, sample_date, force_refresh=False)
if test_result:
    print("Successfully retrieved weather data!")
    print(f"Data contains keys: {list(test_result.keys())}")
    if 'data' in test_result and len(test_result['data']) > 0:
        print(f"Weather conditions: {test_result['data'][0]['weather'][0]['main']} - {test_result['data'][0]['weather'][0]['description']}")
        print(f"Temperature: {test_result['data'][0]['temp']}°C")
        print(f"Humidity: {test_result['data'][0]['humidity']}%")
else:
    print("Failed to retrieve test weather data. Check API key and connection.")

Added coordinates for 20 major airports

Testing with sample airport DEN on 2022-07-22 00:00:00
Loaded cached data for DEN on 2022-07-22
Successfully retrieved weather data!
Data contains keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily']


In [ ]:
# Join weather data with flight data
print("Joining weather data with flight data...")

# First, we need to add weather data for both origin and destination airports
# Create copies with clear column names for origin and destination
origin_weather = weather_df.copy()
origin_weather.columns = ['ORIGIN'] + [col if col == 'FL_DATE' else f'ORIGIN_{col}' 
                                     for col in origin_weather.columns if col != 'AIRPORT']

dest_weather = weather_df.copy()
dest_weather.columns = ['DEST'] + [col if col == 'FL_DATE' else f'DEST_{col}' 
                                  for col in dest_weather.columns if col != 'AIRPORT']

# Now join with flight data
print("Joining origin airport weather...")
enriched_df = flight_data.merge(origin_weather, on=['ORIGIN', 'FL_DATE'], how='left')

print("Joining destination airport weather...")
enriched_df = enriched_df.merge(dest_weather, on=['DEST', 'FL_DATE'], how='left')

# Check missing values (will be missing for airports not in our coordinates list)
print(f"\nEnriched data shape: {enriched_df.shape}")
print(f"Missing weather data for origins: {enriched_df['ORIGIN_TEMP_MAX'].isna().sum()} out of {len(enriched_df)} rows")
print(f"Missing weather data for destinations: {enriched_df['DEST_TEMP_MAX'].isna().sum()} out of {len(enriched_df)} rows")

# Summary of enriched data
print("\nSample of enriched data:")
display(enriched_df.head())

Joining weather data with flight data...


NameError: name 'weather_df' is not defined

In [ ]:
# Save to CSV
output_file = '../data/flights_weather_features.csv'
print(f"Saving enriched data to {output_file}...")
enriched_df.to_csv(output_file, index=False)

print(f"Successfully saved enriched dataset with {len(enriched_df)} rows and {len(enriched_df.columns)} columns")
print(f"New weather features added: {[col for col in enriched_df.columns if 'TEMP' in col or 'PRECIPITATION' in col or 'RAIN' in col or 'SNOW' in col]}")

Saving enriched data to ../data/flights_weather_features.csv...
Successfully saved enriched dataset with 2913802 rows and 19 columns
New weather features added: ['ORIGIN_TEMP_MAX', 'ORIGIN_TEMP_MIN', 'ORIGIN_PRECIPITATION', 'ORIGIN_RAIN', 'ORIGIN_SNOW', 'DEST_TEMP_MAX', 'DEST_TEMP_MIN', 'DEST_PRECIPITATION', 'DEST_RAIN', 'DEST_SNOW']
